In [1]:
import json


user_expectated_formality = {}
expected_formality_distribution = {1: 0, 2: 0, 3: 0, 4: 0}
template_impressions = {"FORMAL": [], "BASE": [], "PERSONAL": [], "FRIENDLY": []}
user_favorite_template = {}
favorite_distribution = {"FORMAL": 0, "BASE": 0, "PERSONAL": 0, "FRIENDLY": 0}
user_template_order = {}
perceived_formality_order = {}
ranking_distribution = {}
user_nodes = {}

black_list = ["66952252774b98ac79dec71a@email.prolific.com", "66347b1d3b38023debe2ddb2"]

# i = 0

def get_style_from_letter(element, user_id):
    if element.upper().strip() == "A":
        return user_template_order[user_id][0]
    elif element.upper().strip() == "B":
        return user_template_order[user_id][1]
    elif element.upper().strip() == "C":
        return user_template_order[user_id][2]
    elif element.upper().strip() == "D":
        return user_template_order[user_id][3]

with open("pilot_log_full.txt", 'r') as log_file:
    for line in log_file:
        # i += 1
        # if i > 20:
        #     break
        user_id = line.split(" ")[1]
        if user_id in black_list:
            continue
        if "PRE-SURVEY:" in line:
            expected = int(line.strip()[-3])
            user_expectated_formality[user_id] = expected
            expected_formality_distribution[expected] += 1
        elif "ORDER" in line:
            order = line.strip().split("ORDER: ")[1][1:-1]
            order = [el.strip("'") for el in order.split(", ")]
            user_template_order[user_id] = order
        elif "NODE_ID" in line:
            node_id = line.split("NODE_ID: ")[1]
            user_nodes[user_id] = node_id
        elif "Preferences:" in line:
            preferences = line.split("Preferences: ")[1]
            preferences = preferences.replace("'", '"')
            preferences = json.loads(preferences)
            impression1 = preferences["impressions1"]
            impression2 = preferences["impressions2"]
            impression3 = preferences["impressions3"]
            impression4 = preferences["impressions4"]
            ranking = preferences["user_ranking"].split(",")
            if len(order) != 4:
                print(user_id, order, len(order))
            else:
                style_ranking = []
                for element in ranking:
                    style = get_style_from_letter(element, user_id)
                    if style is not None:
                        style_ranking.append(style)
                if len(style_ranking) == 4:
                    perceived_formality_order[user_id] = style_ranking
                else:
                    print(f"{user_id}, {ranking}")
                if str(style_ranking) not in ranking_distribution:
                    ranking_distribution[str(style_ranking)] = 0
                ranking_distribution[str(style_ranking)] += 1

            favorite = get_style_from_letter(element=preferences["best_template"], user_id=user_id)
            user_favorite_template[user_id] = favorite
            favorite_distribution[favorite] += 1
            if user_id in user_template_order:
                order = user_template_order[user_id]
                template_impressions[order[0]].append(impression1)
                template_impressions[order[1]].append(impression2)
                template_impressions[order[2]].append(impression3)
                template_impressions[order[3]].append(impression4)


In [2]:
for style in template_impressions:
    with open(f"{style}_impressions.txt", "w") as outfile:
        for impression in template_impressions[style]:
            outfile.write(f"{impression}\n")

In [3]:
print(user_expectated_formality)
print(expected_formality_distribution)
print(user_favorite_template)
print(template_impressions)
print(perceived_formality_order)
print(ranking_distribution)
print(favorite_distribution)

{'665d745228d846bea3c49b0e': 2, '5cfaf97eadb3da00013c1eaa': 2, '5e9d7f715597de04e33079d5': 2, '5efa27ec1c76730eff3dad8e': 3, '611cebd71927a2491a39f942': 2, '669b90a5fd33b2a8e2502f4f': 3, '64c12183ab9cf635c69df81b': 3, '661a51225587ca83a6f23a5c': 3, '5f7b241d63e1da06afec1b30': 2, '669a6464014b2bcfcfecd213': 3, '6102824749c381682b462dfb': 3, '645cf23cf9229b8dd5453a6e': 3, '6151d0a5582003a8d242f7c6': 2, '614765848e321d1e9635a23d': 3, '642959fe58cb911e8d7d644e': 3, '60c919c5d09f37d9ba277afd': 2, '64fead10d35271cb8eafc1c2': 2, '6033b9a844141b0ac09c60cf': 3, '660eda76f3216b44e3783e81': 3, '66742f788c454487df3a1a2a': 2, '650aedb786bfc02e22ec64d6': 1, '669bc3adbc7be494c80aaeae': 3, '64e8b4b5321fee55646ee79f': 3, '660d5600a14b438ee47bbb70': 3, '60db4aed5dd7b87124f51341': 3, '60f3ced4d6311de0d32e6786': 3, '666c5c9b4303c721222ae6d0': 2, '5fa0091ec747500252d2891f': 3, '665c4593e57fa1dd158da1e4': 3, '5bbcf0fc6b9c97000189f6bb': 3, '6672c305775c70ed4186890a': 2, '5e28b99e1aacba092acd8cbd': 3, '66854d

In [36]:
rankings_y = [] # intended order (1-4)
rankings_x = [] # perceived order (1-4)
counts = {"FORMAL": 0, "BASE": 0, "PERSONAL": 0, "FRIENDLY": 0}

for user in perceived_formality_order:
    y = []
    x = [1, 2, 3, 4]
    order = perceived_formality_order[user]
    for condition in order:
        if condition == "FORMAL":
            y.append(1)
            if len(y) == 1:
                counts[condition] += 1
        elif condition == "BASE":
            y.append(2)
            if len(y) == 2:
                counts[condition] += 1
        elif condition == "PERSONAL":
            y.append(3)
            if len(y) == 3:
                counts[condition] += 1
        elif condition == "FRIENDLY":
            y.append(4)
            if len(y) == 4:
                counts[condition] += 1
    rankings_y += y
    rankings_x += x

print(counts)
print(len(rankings_x))
from scipy import stats
res = stats.pearsonr(rankings_x, rankings_y)
print(res)

{'FORMAL': 22, 'BASE': 15, 'PERSONAL': 14, 'FRIENDLY': 19}
112
PearsonRResult(statistic=0.6642857142857143, pvalue=1.4092728133634298e-15)
